# 07. Application + кредитные карты

## Цель

Кратко:
- создать признаки из `credit_card_balance.csv`;
- добавить только карточные признаки к `application_train`;
- обучить CatBoost и записать `application_credit_card`.


## 1. Импорты и пути


In [1]:
from pathlib import Path
import sys


def _is_project_root(path):
    return (
        (path / "src").is_dir()
        and (path / "notebooks").is_dir()
        and (path / "data").is_dir()
    )


project_candidates = [
    Path.cwd(),
    *Path.cwd().parents,
    Path("/content/credit-scoring-system"),
]

if "google.colab" in sys.modules:
    from google.colab import drive

    drive_root = Path("/content/drive/MyDrive")
    if not drive_root.is_dir():
        drive.mount("/content/drive")

    default_drive_project = (
        drive_root / "credit-scoring-system"
    )
    project_candidates.append(default_drive_project)

    if not any(
        _is_project_root(path)
        for path in project_candidates
    ):
        project_candidates.extend(
            config_path.parents[1]
            for config_path in drive_root.rglob("src/config.py")
        )

PROJECT_ROOT = next(
    (
        path.resolve()
        for path in project_candidates
        if _is_project_root(path)
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Не найден корень credit-scoring-system. На Google Drive "
        "должна находиться вся папка проекта с src/, notebooks/ и data/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook_setup import setup_notebook


PROJECT_ROOT = setup_notebook()

Mounted at /content/drive
Installing missing dependency: catboost
Environment: Google Colab
Python: 3.12.13
Project root: /content/drive/MyDrive/credit-scoring-system
Raw data: /content/drive/MyDrive/credit-scoring-system/data/raw
Models: /content/drive/MyDrive/credit-scoring-system/models
Reports: /content/drive/MyDrive/credit-scoring-system/reports


In [2]:
import numpy as np
import pandas as pd
from catboost import (
    CatBoostClassifier,
    Pool,
    cv as catboost_cv,
)
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

from src.config import (
    INTERIM_DATA_DIR as DATA_INTERIM_DIR,
    PROCESSED_DATA_DIR as DATA_PROCESSED_DIR,
    find_data_file,
)
from src.experiment_tracking import save_experiment_result
from src.model_config import (
    get_catboost_device_config,
    get_catboost_gpu_count,
    print_catboost_device_info,
)


APPLICATION_PATH = find_data_file("application_train.csv")
CREDIT_CARD_PATH = find_data_file("credit_card_balance.csv")
CLIENT_SPLIT_PATH = (
    DATA_PROCESSED_DIR / "client_split.csv"
)
CREDIT_CARD_FEATURES_PATH = (
    DATA_INTERIM_DIR / "credit_card_features.csv"
)
RANDOM_STATE = 42
CV_FOLDS = 3


### Устройство CatBoost


In [3]:
gpu_count = get_catboost_gpu_count()
catboost_device_config = get_catboost_device_config(
    gpu_count=gpu_count,
)
print_catboost_device_info(
    catboost_device_config,
    gpu_count=gpu_count,
)


Modeling environment: Google Colab
CatBoost GPU count: 1
CatBoost device: GPU
CatBoost GPU devices: 0


## 2. Загрузка application_train

Основная таблица содержит одну строку на клиента. Техническое значение
`365243` в `DAYS_EMPLOYED` заменяется пропуском.


In [4]:
application = pd.read_csv(APPLICATION_PATH)

if "DAYS_EMPLOYED" in application.columns:
    application["DAYS_EMPLOYED"] = application[
        "DAYS_EMPLOYED"
    ].replace(365243, np.nan)

assert application["SK_ID_CURR"].is_unique
assert application["TARGET"].isin([0, 1]).all()

print("Application:", application.shape)
display(application.head())


Application: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Загрузка credit_card_balance


In [5]:
credit_card = pd.read_csv(CREDIT_CARD_PATH)

assert {"SK_ID_CURR", "SK_ID_PREV"}.issubset(
    credit_card.columns
)

print("Credit card:", credit_card.shape)
display(credit_card.head())


Credit card: (3840312, 23)


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,...,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.0,0.0,0.0,11795.760,...,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,...,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0


## 4. Признаки на уровне месячной записи

Используются доступные месяцы. Загрузка лимита рассчитывается с защитой
от нулевого кредитного лимита.


In [6]:
credit_card = credit_card[
    credit_card["MONTHS_BALANCE"].le(0)
].copy()

credit_card["BALANCE_TO_LIMIT_RATIO"] = (
    credit_card["AMT_BALANCE"]
    / credit_card[
        "AMT_CREDIT_LIMIT_ACTUAL"
    ].replace(0, np.nan)
)

credit_card["HAS_DELINQUENCY"] = (
    credit_card["SK_DPD"].gt(0)
).astype(int)


## 5. Агрегация до клиента


In [7]:
credit_card_features = (
    credit_card
    .groupby("SK_ID_CURR")
    .agg(
        CC_RECORD_COUNT=("MONTHS_BALANCE", "count"),
        CC_BALANCE_MEAN=("AMT_BALANCE", "mean"),
        CC_BALANCE_MAX=("AMT_BALANCE", "max"),
        CC_CREDIT_LIMIT_MEAN=(
            "AMT_CREDIT_LIMIT_ACTUAL",
            "mean",
        ),
        CC_BALANCE_TO_LIMIT_MEAN=(
            "BALANCE_TO_LIMIT_RATIO",
            "mean",
        ),
        CC_BALANCE_TO_LIMIT_MAX=(
            "BALANCE_TO_LIMIT_RATIO",
            "max",
        ),
        CC_PAYMENT_TOTAL=("AMT_PAYMENT_CURRENT", "sum"),
        CC_DRAWING_AMOUNT_MEAN=("AMT_DRAWINGS_CURRENT", "mean"),
        CC_DPD_MEAN=("SK_DPD", "mean"),
        CC_DPD_MAX=("SK_DPD", "max"),
        CC_DELINQUENCY_SHARE=("HAS_DELINQUENCY", "mean"),
    )
    .reset_index()
)

assert credit_card_features["SK_ID_CURR"].is_unique

print(credit_card_features.shape)
display(credit_card_features.head())


(103558, 12)


,SK_ID_CURR,CC_RECORD_COUNT,CC_BALANCE_MEAN,CC_BALANCE_MAX,CC_CREDIT_LIMIT_MEAN,CC_BALANCE_TO_LIMIT_MEAN,CC_BALANCE_TO_LIMIT_MAX,CC_PAYMENT_TOTAL,CC_DRAWING_AMOUNT_MEAN,CC_DPD_MEAN,CC_DPD_MAX,CC_DELINQUENCY_SHARE
0,100006,6,0.000000,0.00,270000.000000,0.000000,0.00000,0.00,0.000000,0.000000,0,0.000000
1,100011,74,54482.111149,189000.00,164189.189189,0.302678,1.05000,358386.75,2432.432432,0.000000,0,0.000000
2,100013,96,18159.919219,161420.22,131718.750000,0.115301,1.02489,688161.24,5953.125000,0.010417,1,0.010417
3,100021,17,0.000000,0.00,675000.000000,0.000000,0.00000,0.00,0.000000,0.000000,0,0.000000
4,100023,8,0.000000,0.00,135000.000000,0.000000,0.00000,0.00,0.000000,0.000000,0,0.000000


## 6. Проверка и сохранение признаков


In [8]:
assert credit_card_features["SK_ID_CURR"].is_unique
assert "TARGET" not in credit_card_features.columns

credit_card_features.to_csv(
    CREDIT_CARD_FEATURES_PATH,
    index=False,
)

print("Сохранено:", CREDIT_CARD_FEATURES_PATH)


Сохранено: /content/drive/MyDrive/credit-scoring-system/data/interim/credit_card_features.csv


## 7. Merge с application


In [9]:
modeling_data = application.merge(
    credit_card_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one",
)

assert len(modeling_data) == len(application)
assert modeling_data["SK_ID_CURR"].is_unique

print("Application:", application.shape)
print("После добавления credit card:", modeling_data.shape)
print(
    "Добавлено признаков:",
    modeling_data.shape[1] - application.shape[1],
)


Application: (307511, 122)
После добавления credit card: (307511, 133)
Добавлено признаков: 11


## Чтение единого client split


In [10]:
if not CLIENT_SPLIT_PATH.exists():
    raise FileNotFoundError(
        "Сначала выполните notebooks/02_application_baseline.ipynb. "
        f"Ожидаемый файл: {CLIENT_SPLIT_PATH}"
    )

client_split = pd.read_csv(CLIENT_SPLIT_PATH)

assert client_split.columns.tolist() == ["SK_ID_CURR", "split"]
assert client_split["SK_ID_CURR"].is_unique
assert set(client_split["split"]) == {"train", "holdout"}
assert set(client_split["SK_ID_CURR"]) == set(application["SK_ID_CURR"])

modeling_data = modeling_data.merge(
    client_split,
    on="SK_ID_CURR",
    how="inner",
    validate="one_to_one",
)

assert len(modeling_data) == len(application)
assert modeling_data["SK_ID_CURR"].is_unique

print(client_split["split"].value_counts())


split
train      246008
holdout     61503
Name: count, dtype: int64


## 9. Создание X и y

`TARGET`, идентификатор клиента и техническая колонка разделения не
передаются модели.


In [11]:
train_data = modeling_data[
    modeling_data["split"].eq("train")
].copy()

n_holdout = int(
    modeling_data["split"].eq("holdout").sum()
)

feature_columns = [
    column
    for column in modeling_data.columns
    if column not in {
        "TARGET",
        "SK_ID_CURR",
        "split",
    }
]

X_train = train_data[feature_columns]
y_train = train_data["TARGET"].astype(int)

assert "TARGET" not in X_train.columns
assert "SK_ID_CURR" not in X_train.columns
assert "split" not in X_train.columns

print("Train:", X_train.shape)
print("Holdout clients (не используется):", n_holdout)


Train: (246008, 131)
Holdout clients (не используется): 61503


## 10. Подготовка данных для CatBoost

Категориальные пропуски заменяются строкой. Числовые `NaN` остаются без
изменений: CatBoost обрабатывает их самостоятельно.


In [12]:
categorical_columns = (
    X_train
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

X_train_catboost = X_train.copy()
X_train_catboost[categorical_columns] = (
    X_train_catboost[categorical_columns]
    .fillna("Unknown")
    .astype(str)
)

print("Категориальных признаков:", len(categorical_columns))


Категориальных признаков: 16


## 11. Стратифицированная кросс-валидация


In [13]:
cv_splitter = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)


## 12. Pool для CatBoost


In [14]:
train_pool = Pool(
    data=X_train_catboost,
    label=y_train,
    cat_features=categorical_columns,
)


## 13. Параметры CatBoost


In [15]:
catboost_params = {
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "custom_metric": ["PRAUC:type=Classic"],
    "auto_class_weights": "Balanced",
    "random_seed": RANDOM_STATE,
    "allow_writing_files": False,
    "verbose": False,
    **catboost_device_config,
}


## 14. Библиотечная CV CatBoost

OOF-предсказания не требуются, поэтому используется `catboost.cv()`
без ручного цикла по фолдам. Test в CV не участвует.


In [16]:
catboost_cv_results = catboost_cv(
    pool=train_pool,
    params=catboost_params,
    folds=cv_splitter,
    early_stopping_rounds=100,
    as_pandas=True,
    verbose=100,
)


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:877: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(
Default metric period is 5 because AUC, PRAUC is/are not implemented for GPU


Training on fold [0/3]


Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.6973384	best: 0.6973384 (0)	total: 173ms	remaining: 2m 52s
100:	test: 0.7455777	best: 0.7455777 (99)	total: 8.87s	remaining: 1m 18s
200:	test: 0.7475272	best: 0.7475272 (200)	total: 18.7s	remaining: 1m 14s
300:	test: 0.7489871	best: 0.7490065 (299)	total: 28.5s	remaining: 1m 6s
400:	test: 0.7517796	best: 0.7517862 (398)	total: 37.3s	remaining: 55.7s
500:	test: 0.7531920	best: 0.7531920 (499)	total: 47s	remaining: 46.8s
600:	test: 0.7542660	best: 0.7542660 (597)	total: 56.7s	remaining: 37.6s
700:	test: 0.7548110	best: 0.7548110 (700)	total: 1m 4s	remaining: 27.6s
800:	test: 0.7551992	best: 0.7551992 (800)	total: 1m 14s	remaining: 18.5s
900:	test: 0.7554535	best: 0.7554539 (897)	total: 1m 23s	remaining: 9.22s
999:	test: 0.7556481	best: 0.7556481 (973)	total: 1m 32s	remaining: 0us
bestTest = 0.7556480765
bestIteration = 973
Training on fold [1/3]


Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.7057562	best: 0.7057562 (0)	total: 159ms	remaining: 2m 38s
100:	test: 0.7493385	best: 0.7493385 (100)	total: 10.5s	remaining: 1m 33s
200:	test: 0.7520750	best: 0.7520750 (200)	total: 20.4s	remaining: 1m 21s
300:	test: 0.7547942	best: 0.7547942 (293)	total: 28.6s	remaining: 1m 6s
400:	test: 0.7560772	best: 0.7560900 (397)	total: 38.3s	remaining: 57.2s
500:	test: 0.7570754	best: 0.7570754 (499)	total: 48.1s	remaining: 47.9s
600:	test: 0.7578558	best: 0.7578810 (593)	total: 56.3s	remaining: 37.4s
700:	test: 0.7586174	best: 0.7586179 (699)	total: 1m 6s	remaining: 28.3s
800:	test: 0.7592518	best: 0.7592518 (800)	total: 1m 15s	remaining: 18.9s
900:	test: 0.7597674	best: 0.7597680 (895)	total: 1m 24s	remaining: 9.33s
999:	test: 0.7598407	best: 0.7598774 (969)	total: 1m 35s	remaining: 0us
bestTest = 0.7598773837
bestIteration = 969
Training on fold [2/3]


Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC:type=Classic is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.7047856	best: 0.7047856 (0)	total: 171ms	remaining: 2m 50s
100:	test: 0.7485615	best: 0.7485615 (100)	total: 10.7s	remaining: 1m 35s
200:	test: 0.7513319	best: 0.7513319 (200)	total: 19.3s	remaining: 1m 16s
300:	test: 0.7526231	best: 0.7526231 (297)	total: 29.1s	remaining: 1m 7s
400:	test: 0.7543467	best: 0.7543467 (395)	total: 38.8s	remaining: 58s
500:	test: 0.7559475	best: 0.7559475 (499)	total: 47.1s	remaining: 46.9s
600:	test: 0.7564266	best: 0.7564266 (600)	total: 57.2s	remaining: 37.9s
700:	test: 0.7574002	best: 0.7574002 (700)	total: 1m 7s	remaining: 28.8s
800:	test: 0.7580359	best: 0.7580359 (799)	total: 1m 15s	remaining: 18.8s
900:	test: 0.7583640	best: 0.7583746 (892)	total: 1m 25s	remaining: 9.44s
999:	test: 0.7586233	best: 0.7586233 (999)	total: 1m 36s	remaining: 0us
bestTest = 0.758623302
bestIteration = 999


## 15. Лучшая итерация и CV-метрики


In [17]:
auc_mean_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-AUC")
    and column.endswith("-mean")
)

auc_std_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-AUC")
    and column.endswith("-std")
)

pr_auc_mean_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-PRAUC")
    and column.endswith("-mean")
)

pr_auc_std_column = next(
    column
    for column in catboost_cv_results.columns
    if column.startswith("test-PRAUC")
    and column.endswith("-std")
)

best_cv_index = catboost_cv_results[
    auc_mean_column
].idxmax()

best_cv_row = catboost_cv_results.loc[
    best_cv_index
]

best_iteration = int(
    best_cv_row["iterations"]
) + 1

cv_roc_auc = float(
    best_cv_row[auc_mean_column]
)

cv_roc_auc_std = float(
    best_cv_row[auc_std_column]
)

cv_pr_auc = float(
    best_cv_row[pr_auc_mean_column]
)

cv_pr_auc_std = float(
    best_cv_row[pr_auc_std_column]
)

print(f"Лучшая итерация: {best_iteration}")
print(
    f"CV ROC-AUC: {cv_roc_auc:.4f} "
    f"± {cv_roc_auc_std:.4f}"
)
print(
    f"CV PR-AUC: {cv_pr_auc:.4f} "
    f"± {cv_pr_auc_std:.4f}"
)


Лучшая итерация: 1000
CV ROC-AUC: 0.7580 ± 0.0022
CV PR-AUC: nan ± nan


## 16. Итоговая модель CatBoost


In [18]:
catboost_model = CatBoostClassifier(
    iterations=best_iteration,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=RANDOM_STATE,
    allow_writing_files=False,
    verbose=100,
    **catboost_device_config,
)


## 17. Обучение итоговой модели


In [19]:
catboost_model.fit(
    X_train_catboost,
    y_train,
    cat_features=categorical_columns,
)


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 73.6ms	remaining: 1m 13s
100:	total: 4.51s	remaining: 40.2s
200:	total: 10.6s	remaining: 42.2s
300:	total: 14.9s	remaining: 34.7s
400:	total: 19.3s	remaining: 28.8s
500:	total: 25.3s	remaining: 25.2s
600:	total: 29.7s	remaining: 19.7s
700:	total: 34s	remaining: 14.5s
800:	total: 39.8s	remaining: 9.89s
900:	total: 44.1s	remaining: 4.85s
999:	total: 48.4s	remaining: 0us


CatBoostClassifier(allow_writing_files=False, auto_class_weights='Balanced', depth=6, devices='0', eval_metric='AUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', random_seed=42, task_type='GPU', verbose=100)

## Запись CV-результата


In [20]:
current_result = {
    "experiment": "application_credit_card",
    "notebook": "07_credit_card_features.ipynb",
    "model": "CatBoostClassifier",
    "feature_set": "application + credit_card",
    "source_tables": "application_train.csv, credit_card_balance.csv",
    "device": catboost_device_config["task_type"],
    "n_train": len(train_data),
    "n_holdout": n_holdout,
    "n_features": X_train.shape[1],
    "cv_folds": CV_FOLDS,
    "best_iteration": best_iteration,
    "cv_roc_auc": cv_roc_auc,
    "cv_roc_auc_std": cv_roc_auc_std,
    "cv_pr_auc": cv_pr_auc,
    "cv_pr_auc_std": cv_pr_auc_std,
    "holdout_roc_auc": None,
    "holdout_pr_auc": None,
}

all_results = save_experiment_result(current_result)
display(all_results)


,experiment,notebook,model,feature_set,source_tables,device,n_train,n_holdout,n_features,cv_folds,best_iteration,cv_roc_auc,cv_roc_auc_std,cv_pr_auc,cv_pr_auc_std,holdout_roc_auc,holdout_pr_auc
0,application_logistic,02_application_baseline.ipynb,LogisticRegression,application,application_train.csv,CPU,246008,61503,120,3,NaN,0.744841,0.002434,0.217865,0.005206,NaN,NaN
1,application_catboost,02_application_baseline.ipynb,CatBoostClassifier,application,application_train.csv,GPU,246008,61503,120,3,1000.0,0.754176,0.002317,NaN,NaN,NaN,NaN
2,application_bureau,03_bureau_features.ipynb,CatBoostClassifier,application + bureau,"application_train.csv, bureau.csv, bureau_bala...",GPU,246008,61503,134,3,1000.0,0.758362,0.001620,NaN,NaN,NaN,NaN
3,application_previous,04_previous_application_features.ipynb,CatBoostClassifier,application + previous_application,"application_train.csv, previous_application.csv",GPU,246008,61503,131,3,1000.0,0.760286,0.002017,NaN,NaN,NaN,NaN
4,application_installments,05_installments_features.ipynb,CatBoostClassifier,application + installments,"application_train.csv, installments_payments.csv",GPU,246008,61503,129,3,1000.0,0.759957,0.000499,NaN,NaN,NaN,NaN
5,application_pos_cash,06_pos_cash_features.ipynb,CatBoostClassifier,application + POS_CASH,"application_train.csv, POS_CASH_balance.csv",GPU,246008,61503,127,3,996.0,0.761249,0.001983,NaN,NaN,NaN,NaN
6,application_credit_card,07_credit_card_features.ipynb,CatBoostClassifier,application + credit_card,"application_train.csv, credit_card_balance.csv",GPU,246008,61503,131,3,1000.0,0.758037,0.002157,NaN,NaN,NaN,NaN


## Выводы


In [21]:
print("Эксперимент: application + credit_card_balance")
print(f"Количество признаков: {X_train.shape[1]}")
print(f"Лучшая итерация: {best_iteration}")
print(f"CV ROC-AUC: {cv_roc_auc:.4f}")
print(f"CV PR-AUC: {cv_pr_auc:.4f}")
print("Holdout не использовался: результат сравнивается только по CV.")


Эксперимент: application + credit_card_balance
Количество признаков: 131
Лучшая итерация: 1000
CV ROC-AUC: 0.7580
CV PR-AUC: nan
Holdout не использовался: результат сравнивается только по CV.
